# Ref. Res. Dev Notebook:

Same method as naive method 1, but padded to max of 42 objects.

In [1]:
import random
import torch
import io
import pyarrow as pa
import os
import copy
import pytorch_lightning as pl
from sacred import Experiment
from PIL import Image
from tqdm import tqdm
import numpy as np
import skimage.io as skio
import matplotlib.pyplot as plt
from refer import REFER

from torch.optim import AdamW

from transformers import ElectraTokenizer

from refcoco_utils import get_bounded_subimage
from refcoco_utils import _config
from refcoco_utils import _loss_names

from meter.transforms import keys_to_transforms
from meter.config import ex
from meter.modules import METERTransformerSS
from meter.datamodules.multitask_datamodule import MTDataModule
from meter.datasets.base_dataset import BaseDataset

## Data

In [2]:
data_root = '/home/claytonfields/nlp/code/data/coco'  # contains refclef, refcoco, refcoco+, refcocog and images
dataset = 'refcoco' 
splitBy = 'unc'
refer = REFER(data_root, dataset, splitBy)

loading dataset refcoco into memory...
testing
creating index...
index created.
DONE (t=9.87s)


## Model

In [3]:
_config = copy.deepcopy(_config)
pl.seed_everything(_config["seed"])

dm = MTDataModule(_config, dist=False)
model = METERTransformerSS(_config)

Global seed set to 0
Some weights of the model checkpoint at google/electra-small-discriminator were not used when initializing ElectraModel: ['discriminator_predictions.dense_prediction.weight', 'discriminator_predictions.dense.weight', 'discriminator_predictions.dense_prediction.bias', 'discriminator_predictions.dense.bias']
- This IS expected if you are initializing ElectraModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ElectraModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


## Data Class

In [154]:
class RefcocoDataset(torch.utils.data.Dataset):

    def __init__(self, refer, tokenizer, split='', max_bb = 42):
        self.tokenizer = tokenizer
        self.refer = refer
        self.max_bb = max_bb
        self.split = split
        self.sent_ids = self.get_sent_ids()
        self.duds = []

    def __len__(self):
        return len(self.sent_ids)
    
    def get_sent_ids(self):
        sent_ids = []
        for ref_id in self.refer.getRefIds(split=self.split):
            ref = self.refer.Refs[ref_id]
            img_id = ref['image_id']
            objs = self.refer.imgToAnns[img_id]
            if len(objs) <= self.max_bb:
                for sent_id in ref['sent_ids']:
                    sent_ids.append(sent_id)
        return sent_ids
    
    def __getitem__(self, index):
        sent_id = self.sent_ids[index]
        ref = self.refer.sentToRef[sent_id]
        sent = self.refer.Sents[sent_id]
        
        img_id = ref['image_id']
        ann_id = ref['ann_id']
        
        objs = self.refer.imgToAnns[img_id]
        obj_ids = [obj['id'] for obj in objs]
        obj_pad = [0 for _ in range(self.max_bb-len(obj_ids))]
        obj_ids_total = obj_ids+obj_pad

        sub_images = []
        for obj in objs:
            try:
                x_a = get_bounded_subimage(self.refer, img_id, obj['id'], xs=224,ys=224, show=False)
            except ValueError:
                print(f'ValueError at setence id: {sent_id}')
                self.duds.append(sent_id)
                break
                
                if x_a is not None:
                    sub_images.append(x_a)
        
        num_sub_images = len(sub_images)
        num_pad = self.max_bb - num_sub_images 
        
        pad_image = torch.zeros(1,3,224,224)
        for _ in range(num_pad):
            sub_images.append(pad_image)
        
        # text ids
        ids = self.tokenizer.encode(
            sent['sent'],
            padding="max_length",
            truncation=True,
            max_length=40,
            return_special_tokens_mask=True,
        )
        repeat_ids = torch.tensor(ids).repeat(num_sub_images,1)
        pad_ids =  torch.zeros(num_pad,40)
        text_ids = torch.cat((repeat_ids, pad_ids)).to(torch.long)
        # text masks
        num_tokens = torch.where(text_ids[0] > 0)[0].size(dim=0)
        masks = torch.cat((torch.ones(num_tokens), torch.zeros(40-num_tokens))).to(torch.long)
        repeat_masks = masks.repeat(num_sub_images,1)
        pad_masks = torch.zeros(num_pad, 40)
        text_masks = torch.cat((repeat_masks, pad_masks)).to(torch.long)
        # text_labels
        labels = torch.full((40,),-100)
        repeat_labels = labels.repeat(num_sub_images, 1)
        pad_labels = torch.zeros(num_pad, 40)
        text_labels = torch.cat((repeat_labels, pad_labels)).to(torch.long)
        
        target = torch.tensor([obj_ids.index(ann_id)]).to(self.device)
        
        return_dict = {
            'ann_id' : ann_id,
            'image' : [torch.cat(sub_images).to(self.device)],
            'obj_ids' : torch.tensor(obj_ids_total).to(self.device),
            'sent_id' : sent_id,
            'target' : target,
            'text' : sent['sent'],
            'text_ids' : text_ids.to(self.device),
            'text_labels' : text_labels.to(self.device),
            'text_masks' : text_masks.to(self.device)
        }
        return return_dict

This function simply returns a list of examples in a dictionary. This will work for now as a simple solution. More will probably need to be done later. 

In [129]:
def collate_fn(batch):
    return batch

## Ref Res with Meter

In [139]:
# device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
device = torch.device('cpu')
optimizer = AdamW(model.parameters(), lr=1e-4)
loss_fn = torch.nn.functional.cross_entropy
# Ref Res with METER
tokenizer = ElectraTokenizer.from_pretrained('google/electra-small-discriminator')
BATCH_SIZE = 10
epochs = 1

In [140]:
train_ds = RefcocoDataset(refer, tokenizer, split='train')
train_params = {'batch_size': BATCH_SIZE,
                'shuffle': False,
                'num_workers': 0,
                'collate_fn' : collate_fn
                }

train_loader = torch.utils.data.DataLoader(ds, **train_params)

In [141]:
for i, data in enumerate(training_loader):
    if i ==1:
        break
data[0]['image'][0].shape

torch.Size([5, 3, 224, 224])

In [152]:
train_ds[10]

{'ann_id': 485695,
 'image': [tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
            [0., 0., 0.,  ..., 0., 0., 0.],
            [0., 0., 0.,  ..., 0., 0., 0.],
            ...,
            [0., 0., 0.,  ..., 0., 0., 0.],
            [0., 0., 0.,  ..., 0., 0., 0.],
            [0., 0., 0.,  ..., 0., 0., 0.]],
  
           [[0., 0., 0.,  ..., 0., 0., 0.],
            [0., 0., 0.,  ..., 0., 0., 0.],
            [0., 0., 0.,  ..., 0., 0., 0.],
            ...,
            [0., 0., 0.,  ..., 0., 0., 0.],
            [0., 0., 0.,  ..., 0., 0., 0.],
            [0., 0., 0.,  ..., 0., 0., 0.]],
  
           [[0., 0., 0.,  ..., 0., 0., 0.],
            [0., 0., 0.,  ..., 0., 0., 0.],
            [0., 0., 0.,  ..., 0., 0., 0.],
            ...,
            [0., 0., 0.,  ..., 0., 0., 0.],
            [0., 0., 0.,  ..., 0., 0., 0.],
            [0., 0., 0.,  ..., 0., 0., 0.]]],
  
  
          [[[0., 0., 0.,  ..., 0., 0., 0.],
            [0., 0., 0.,  ..., 0., 0., 0.],
            [0., 0., 0., 

**Training**

In [143]:
def train(model, training_ds, optimizer, loss_fn, device):
    model.to(device)
    model.train()
    losses = []
    for data in tqdm(train_ds):
        if data['sent_id'] in train_ds.duds:
            continue
        try:
            optimizer.zero_grad()

            sent_id = data['sent_id']
            infer_dict = model.infer(data)
            logits = model.ref_classifier(infer_dict['cls_feats'])

            obj_ids = data['obj_ids']
            ann_id = data['ann_id']

            target = data['target']
            loss = loss_fn(logits.reshape(1,-1),target)
            losses.append(loss.item())
            loss.backward()

            optimizer.step()
        except RuntimeError:
            print(f'Runtime Error at sent_id = {sent_id}')
            training_ds.duds.append(sent_id)
    return losses, loss

### Eval

In [144]:
eval_ds = RefcocoDataset(refer, tokenizer, split='val')
eval_params = {'batch_size': BATCH_SIZE,
                'shuffle': False,
                'num_workers': 0,
                'collate_fn' : collate_fn
                }

eval_loader = torch.utils.data.DataLoader(ds, **eval_params)

**Eval Loop**

In [145]:
def evaluate(model, eval_ds, device):
    model.to(device)
    gold = []
    with torch.no_grad():
        for data in tqdm(eval_ds):
            if data['sent_id'] in eval_ds.duds:
                continue
            try:
                sent_id = data['sent_id']
                infer_dict = model.infer(data)
                logits = model.ref_classifier(infer_dict['cls_feats'])

                obj_ids = data['obj_ids']
                ann_id = data['ann_id']

                pred_index = logits.argmax()
                pred_id = obj_ids[pred_index.item()]
                if pred_id.item() == ann_id:
                    gold.append(1)
                else:
                    gold.append(0)
            except RuntimeError:
                print(f'RuntimeError at sent_id = {sent_id}')
                eval_ds.duds.append(sent_id)
    return gold

In [153]:
with open('eval.txt','w') as f:
    for epoch in range(epochs):
        losses, loss = train(model, train_ds, optimizer, loss_fn, device)
        avg_loss = np.average(losses)
        loss_string = f'Epoch: {epoch}, Final Loss: {loss.item()}, Average Loss: {avg_loss} \n'
        f.write(loss_string)
        print(loss_string)  
        pd.DataFrame(losses, columns=['Loss']).to_csv(f'Epoch_{epoch}_losses.csv')
        
        gold = evaluate(model, eval_ds, device)
        acc = np.average(gold)
        acc_string = f'Epoch: {epoch}, Acurracy on test set: {acc} \n'
        f.write(acc_string)
        print(acc_string)
pd.DataFrame(train_ds.duds, columns=['Sent ID']).to_csv('TrainingErrors.csv')
pd.DataFrame(eval_ds.duds, columns=['Sent ID']).to_csv('EvalErrors.csv')

  0%|                                    | 15/119707 [00:38<86:24:50,  2.60s/it]


KeyboardInterrupt: 